In [ ]:
import yaml
import os

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

In [ ]:
import faiss
print(faiss.get_num_gpus())  # 返回 GPU 数量，如果是 CPU 版本会报错或返回 0
print(faiss.StandardGpuResources)  # 如果可以导入，说明支持 GPU
print(faiss.__version__)

In [ ]:
from openai import OpenAI

# 指向本地 vLLM 服务
client = OpenAI(base_url="http://localhost:8000/v1", api_key="my_secret_key")

response = client.chat.completions.create(
    model="Qwen3-4B-Instruct-2507",
    messages=[
        {"role": "system", "content": "你是一个有帮助的助手"},
        {"role": "user", "content": "介绍一下你自己"}
    ],
    temperature=0.7,
    max_tokens=128
)

print(response.choices[0].message.content)

In [ ]:
from vllm import LLM, SamplingParams
import os
import multiprocessing as mp
    
mp.set_start_method("spawn", force=True)  # ✅ 推荐的启动方法

# 指定要用的 GPU，例如 0,1 两张卡
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

# 打开 vLLM 的多进程模式
# os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


# 1. 初始化vLLM，加载本地Qwen3模型
model_path = "./models/Qwen3-4B-Instruct-2507"  # 修改为你本地路径
llm = LLM(model=model_path, dtype="float16", tensor_parallel_size=4, gpu_memory_utilization=0.75)

# 2. 设置生成参数
params = SamplingParams(
        temperature=0.7,   # 控制随机性
        top_p=0.8,         # nucleus sampling
        top_k=20,          # top-k采样
        min_p=0.0,         # 最小概率
        max_tokens=16384,  # 输出长度，官方推荐16k
        presence_penalty=0.0  # 可根据需要调节0-2，避免重复
)

# 3. 简单生成示例
prompt = "请用简短的语言解释量子力学的基本概念。"

# vLLM生成
for result in llm.generate(prompt, sampling_params=params):
        print(result.text)

# 4. 关闭模型释放资源
llm.close()

In [ ]:
from client import Client
from QwenEmbeddings import QwenEmbeddings
from QwenRerankers import QwenReranker
import os
# from vllm.distributed.parallel_state import destroy_model_parallel
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5"

embedding = QwenEmbeddings(model_name="./models/qwen3-embedding-4b", device="cuda:3")
reranker = QwenReranker()

client = Client(embedding=embedding, reranker=reranker, vectorstore_path="common_sense_db_4b")
client.load_vectorstore()

In [ ]:
query = "What is the capital of China?"
results, q_v = client.retrieve(query)

In [ ]:
results[0].document.page_content

In [ ]:
import torch
print(torch.version.cuda)
print(torch.__version__)

In [ ]:
from QwenEmbeddings import QwenEmbeddings

embedding = QwenEmbeddings(model_name="./models/qwen3-embedding-4b", device="cuda:3")

q_vec = QwenEmbeddings().embed_query("hello world")
print(type(q_vec), q_vec.shape)

In [ ]:
from QwenRerankers import QwenReranker
import os
from vllm.distributed.parallel_state import destroy_model_parallel

# os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3"

reranker = QwenReranker()

task = 'Given a web search query, retrieve relevant passages that answer the query'
queries = ["What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

pairs = list(zip(queries, documents))
inputs = reranker.process_inputs(pairs)
scores = reranker.compute_logits(inputs)
print('scores', scores)

destroy_model_parallel()

In [ ]:
from QwenRerankers import QwenReranker
reranker = QwenReranker()
pairs = [reranker.format_instruction(task, query, doc) for query, doc in zip(queries, documents)]

In [ ]:
# Tokenize the input texts
inputs = reranker.process_inputs(pairs)
scores = reranker.compute_logits(inputs)

In [ ]:
print("scores: ", scores)

In [ ]:
with open("prompt.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

system_msg = config["system_prompt"]["content"]

In [ ]:
# 4.加载测试数据集，这里以trivia_qa为例子，这里取前100个
from datasets import Dataset, DatasetDict, load_dataset
from pathlib import Path

root_dir = "./test_dataset/natural_questions/default"
parquet_files = []

# 递归查找所有符合的文件
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        parquet_files.append(os.path.join(dirpath, filename))

print(f"找到 {len(parquet_files)} 个 validation parquet 文件")

# 加载所有文件为一个dataset列表
datasets_list = load_dataset("parquet", data_files={"validation": parquet_files}, split="validation")

In [ ]:
datasets_list[0]['document']['html']

In [ ]:
import validation_tools
text = validation_tools.strip_html(datasets_list[0]['document']['html'])

In [ ]:
text

In [ ]:
# 抽样选取 100 条（打乱顺序）
# random_samples = datasets_list.shuffle(seed=42).select(range(100))
import validation_tools
random_samples = []
for i, sample in enumerate(datasets_list):
    background, question, answer = validation_tools.get_natural_questions(sample)
    print(background)
    if len(answer) > 0:
        random_samples.append(sample)
    i += 1
    if i == 100:
        break

In [ ]:
system_msg

In [ ]:
general_few_shot = config["fewshots"]["general"]["examples"]["content"]

In [ ]:
general_few_shot

In [ ]:
user_msg_template = config["user_prompts"]["template"]

In [ ]:
user_msg = user_msg_template.format(query="query", contexts_block="contexts_block", instruction="background")


In [ ]:
user_msg